# Tensor basics

A short tour of `cmtoolkit.tensor_algebra` for second- and fourth-order tensors in a 3D orthonormal basis.
Each identity is derived **symbolically** (sympy) and, where useful, cross-checked **numerically** (numpy).
Notation: $\boldsymbol A\cdot\boldsymbol B = A_{ij}B_{ij}$, $\mathbb C[\boldsymbol A] = C_{ijkl}A_{kl}\,\boldsymbol e_i\otimes\boldsymbol e_j$.

In [1]:
import numpy as np
import sympy as sp

import cmtoolkit.tensor_algebra as ta
from cmtoolkit.checks import assert_close, assert_symbolic_zero, numerical_gradient, show
from cmtoolkit.elasticity import compliance_isotropic, stiffness_isotropic
from cmtoolkit.kinematics import polar_decomposition
from cmtoolkit.tensor_analysis import d_normalized_tensor, d_third_invariant

## 1. Additive decompositions and the Cayley–Hamilton theorem

$\boldsymbol A = \operatorname{sym}\boldsymbol A + \operatorname{skw}\boldsymbol A = \operatorname{sph}\boldsymbol A + \operatorname{dev}\boldsymbol A$,
and the two parts of each split are orthogonal. For a completely generic $\boldsymbol A$ (nine independent symbols)
$\boldsymbol A^3 - I_A\boldsymbol A^2 + II_A\boldsymbol A - III_A\boldsymbol I = \boldsymbol 0$ holds identically.

In [2]:
A = ta.symbolic_tensor("A")
assert_symbolic_zero(ta.sym(A) + ta.skw(A) - A, "A = sym A + skw A")
assert_symbolic_zero(ta.einsum("ij,ij->", ta.sym(A), ta.skw(A)), "sym A · skw A = 0")
assert_symbolic_zero(ta.einsum("ij,ij->", ta.sph(A), ta.dev(A)), "sph A · dev A = 0")
I_A, II_A, III_A = ta.principal_invariants(A)
show(r"II_A", sp.expand(II_A))
assert_symbolic_zero(ta.cayley_hamilton_residual(A), "Cayley-Hamilton for a generic A", simplifier=sp.expand)

[OK] A = sym A + skw A
[OK] sym A · skw A = 0
[OK] sph A · dev A = 0


<IPython.core.display.Math object>

[OK] Cayley-Hamilton for a generic A


## 2. Eigenprojectors and the polar decomposition

A symmetric tensor is $\boldsymbol A = \sum_i \lambda_i\boldsymbol P_i$ with one projector per *distinct* eigenvalue,
so a repeated eigenvalue gives a projector of rank 2. Isotropic tensor functions $f(\boldsymbol A) = \sum_i f(\lambda_i)\boldsymbol P_i$
give $\boldsymbol U = \sqrt{\boldsymbol C}$ and hence $\boldsymbol F = \boldsymbol R\boldsymbol U = \boldsymbol V\boldsymbol R$.
For simple shear with $\gamma = 3/2$ the decomposition is exact in rational numbers.

In [3]:
S = sp.Matrix([[2, 1, 0], [1, 2, 0], [0, 0, 3]])            # eigenvalues 1, 3, 3
pairs = ta.eigenprojectors(S)
for lam, P in pairs:
    show(rf"\lambda = {lam}:\quad \boldsymbol P", P)
assert_symbolic_zero(sum((P for _, P in pairs), sp.zeros(3, 3)) - sp.eye(3), "sum of projectors = I")

F = sp.Matrix([[1, sp.Rational(3, 2), 0], [0, 1, 0], [0, 0, 1]])   # simple shear, gamma = 3/2
R, U, V = polar_decomposition(F)
show(r"\boldsymbol R", R)
show(r"\boldsymbol U", U)
assert_symbolic_zero(R @ U - F, "F = R U")
assert_symbolic_zero(V @ R - F, "F = V R")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

[OK] sum of projectors = I


<IPython.core.display.Math object>

<IPython.core.display.Math object>

[OK] F = R U
[OK] F = V R


## 3. Rotations

Euler–Rodrigues: $\boldsymbol Q(\boldsymbol n,\varphi) = \cos\varphi\,\boldsymbol I + (1-\cos\varphi)\,\boldsymbol n\otimes\boldsymbol n + \sin\varphi\,[\boldsymbol n]_\times$.
Axis and angle are recovered from $\cos\varphi = (\operatorname{tr}\boldsymbol Q - 1)/2$ and $\operatorname{skw}\boldsymbol Q = \sin\varphi\,[\boldsymbol n]_\times$.

In [4]:
phi = sp.Symbol("varphi", real=True)
show(r"\boldsymbol Q(\boldsymbol e_3, \varphi)", ta.rotation_tensor(sp.Matrix([0, 0, 1]), phi))

n = np.array([1.0, 2.0, 2.0])
Q = ta.rotation_tensor(n, 2.0)
axis, angle = ta.rotation_axis_angle(Q)
assert_close(Q.T @ Q, np.eye(3), label="Q^T Q = I")
assert_close(axis, n / 3, label="recovered axis")
assert_close(angle, 2.0, label="recovered angle")

<IPython.core.display.Math object>

[OK] Q^T Q = I: max |Δ| = 2.2e-16
[OK] recovered axis: max |Δ| = 5.6e-17
[OK] recovered angle: max |Δ| = 0.0e+00


## 4. Isotropic fourth-order tensors and Mandel notation

The identity on Lin splits into three orthogonal projectors, $\mathbb I = \mathbb P_\mathrm{sph} + \mathbb P_\mathrm{dev} + \mathbb P_\mathrm{skw}$,
with eigenspace dimensions 1, 5 and 3. The isotropic stiffness is $\mathbb C = 3K\,\mathbb P_\mathrm{sph} + 2G\,\mathbb P_\mathrm{dev}$,
so its inverse on Sym is $\mathbb S = \frac{1}{3K}\mathbb P_\mathrm{sph} + \frac{1}{2G}\mathbb P_\mathrm{dev}$.
Mandel notation maps Sym to $\mathbb R^6$ while preserving the scalar product; inverting $\mathbb C$ on Sym then becomes a 6×6 matrix inverse.

In [5]:
P_sph, P_dev, P_skw = ta.I_sph(), ta.I_dev(), ta.I_skw()
assert_symbolic_zero(P_sph + P_dev + P_skw - ta.identity4(), "I_sph + I_dev + I_skw = identity")
for name, P in (("sph", P_sph), ("dev", P_dev), ("skw", P_skw)):
    assert_symbolic_zero(ta.compose4(P, P) - P, f"I_{name} idempotent, dimension {ta.einsum('ijij->', P)}")

lam, mu = sp.symbols("lambda mu", positive=True)
C = stiffness_isotropic(lam, mu)
show(r"[\mathbb C]_\mathrm{Mandel}", ta.to_mandel_matrix(C))

[OK] I_sph + I_dev + I_skw = identity
[OK] I_sph idempotent, dimension 1
[OK] I_dev idempotent, dimension 5


[OK] I_skw idempotent, dimension 3

<IPython.core.display.Math object>

In [6]:
S_mandel = ta.to_mandel_matrix(compliance_isotropic(lam, mu))
assert_symbolic_zero(ta.to_mandel_matrix(C).inv() - S_mandel, "[C]^-1 = [S]")
eps = ta.symbolic_tensor("varepsilon", symmetric=True)
assert_symbolic_zero(ta.to_mandel_matrix(C) @ ta.to_mandel_vector(eps) - ta.to_mandel_vector(ta.linear_map(C, eps)),
                     "[C][eps] = [C[eps]]")

[OK] [C]^-1 = [S]
[OK] [C][eps] = [C[eps]]


## 5. Generalized gradients

$\partial III_A/\partial\boldsymbol A = \operatorname{cof}\boldsymbol A$, and for the von Mises flow direction
$\boldsymbol N' = \boldsymbol\sigma'/\|\boldsymbol\sigma'\|$:
$\partial\boldsymbol N'/\partial\boldsymbol\sigma = (\mathbb P_\mathrm{dev} - \boldsymbol N'\otimes\boldsymbol N')/\|\boldsymbol\sigma'\|$.
Both are checked symbolically or against central differences.

In [7]:
d_det = sp.Matrix(3, 3, lambda i, j: sp.diff(ta.det(A), A[i, j]))
assert_symbolic_zero(d_det - d_third_invariant(A), "d det(A)/dA = cof A (symbolic)", simplifier=sp.expand)

rng = np.random.default_rng(0)
sigma = ta.sym(rng.normal(size=(3, 3)))
flow_direction = lambda X: ta.dev(ta.sym(X)) / ta.norm(ta.dev(ta.sym(X)))
assert_close(numerical_gradient(flow_direction, sigma), d_normalized_tensor(sigma, deviatoric=True), tol=1e-7,
             label="dN'/dsigma vs. central differences")

[OK] d det(A)/dA = cof A (symbolic)
[OK] dN'/dsigma vs. central differences: max |Δ| = 4.3e-11
